In [10]:
import sys
from pathlib import Path

def find_data_folder():
    """DATA 폴더 찾기"""
    current = Path.cwd()

    # 상위 5단계까지 검색
    for i in range(5):
        data_path = current / "DATA"
        if data_path.exists():
            target_files = [
                data_path / "stock_invest_function.py",
                data_path / "us_stock_valuation_preprocessing.py"
            ]
            if all(f.exists() for f in target_files):
                return data_path
        current = current.parent
    return None

def fix_and_import_modules():
    """모듈을 수정해서 import"""
    data_path = find_data_folder()
    if not data_path:
        print("DATA 폴더를 찾을 수 없습니다")
        return None, None

    sys.path.insert(0, str(data_path))
    print(f"DATA 폴더 발견: {data_path}")

    # stock_invest_function은 정상 import
    try:
        import stock_invest_function as stock_func
        print("stock_invest_function import 성공!")
    except Exception as e:
        print(f"stock_invest_function import 실패: {e}")
        stock_func = None

    # us_stock_valuation_preprocessing은 파일을 읽어서 수정 후 실행
    try:
        preprocessing_file = data_path / "us_stock_valuation_preprocessing.py"

        # 파일 내용 읽기
        with open(preprocessing_file, 'r', encoding='utf-8') as f:
            content = f.read()

        # 문제가 되는 부분을 찾아서 수정
        # "from stock_invest_function import *"를 모듈 최상위로 이동
        lines = content.split('\n')

        # import * 구문을 찾아서 제거하고 최상위에 추가
        fixed_lines = []
        import_lines = []
        inside_function = False

        for line in lines:
            # 함수 시작 감지
            if line.strip().startswith('def '):
                inside_function = True

            # import * 구문 처리
            if 'from stock_invest_function import *' in line:
                if inside_function:
                    # 함수 내부에 있으면 제거
                    continue
                else:
                    # 최상위에 있으면 그대로 유지
                    import_lines.append(line)
                    continue

            fixed_lines.append(line)

        # 수정된 내용 조합
        final_content = '\n'.join(import_lines + [''] + fixed_lines)

        # 수정된 내용을 실행해서 모듈 생성
        import types
        stock_prep = types.ModuleType('us_stock_valuation_preprocessing')

        # get_db_host 함수가 필요한 경우를 위해 미리 정의
        if stock_func and hasattr(stock_func, 'get_db_host'):
            stock_prep.get_db_host = stock_func.get_db_host
        else:
            stock_prep.get_db_host = lambda: 'localhost'

        # 수정된 코드 실행
        exec(final_content, stock_prep.__dict__)

        print("us_stock_valuation_preprocessing 수정 후 import 성공!")
        return stock_func, stock_prep

    except Exception as e:
        print(f"파일 수정 중 오류: {e}")
        return stock_func, None

# 실행
stock_func, stock_prep = fix_and_import_modules()

if stock_func and stock_prep:
    print("모든 모듈이 성공적으로 로드되었습니다!")
    print("사용 가능한 함수들:")
    if hasattr(stock_prep, 'run_data_preprocessing'):
        print("- stock_prep.run_data_preprocessing()")
    if hasattr(stock_prep, 'run_simple_preprocessing'):
        print("- stock_prep.run_simple_preprocessing()")
else:
    print("모듈 로드에 실패했습니다.")

DATA 폴더 발견: C:\Users\MetaM\PycharmProjects\stock_forecast\DATA
stock_invest_function import 성공!
us_stock_valuation_preprocessing 수정 후 import 성공!
모든 모듈이 성공적으로 로드되었습니다!
사용 가능한 함수들:
- stock_prep.run_data_preprocessing()
- stock_prep.run_simple_preprocessing()


In [11]:
from DATA.stock_invest_function import *

In [12]:
api_key = 'hT0gAk87j9xZx4PlBApvBqfVL5IahvgV'

db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

In [13]:
# 간단 사용
# result = stock_prep.run_data_preprocessing(['SMCI'], api_key)

# 수출 데이터 포함
result = stock_prep.run_data_preprocessing(['SMCI'], api_key, '851762', db_info)

# 단일 종목 간단 버전
# result = stock_prep.run_simple_preprocessing('AAPL', api_key)

데이터 전처리 시작
대상 종목: SMCI
HS Code: 851762
API 연결 테스트 중...
API 연결 성공
수출 데이터 수집 중... (HS Code: 851762)
가장 최근 input_date: 2025-09-02
해당 날짜의 데이터: 165개
수출 데이터 수집 완료: 165 레코드
매출 데이터 수집 중...


Revenue:   0%|          | 0/1 [00:00<?, ?it/s]

   SMCI: 84개 분기


Revenue: 100%|██████████| 1/1 [00:01<00:00,  1.41s/it]


매출 데이터 수집 완료: 84 레코드
시가총액 데이터 수집 중...


Market Cap: 100%|██████████| 1/1 [00:19<00:00, 19.23s/it]

   SMCI: 189개 월
시가총액 데이터 수집 완료: 189 레코드
매출 및 시가총액 데이터 결합 중...
데이터 결합 완료: 187 레코드
PSR 계산 중...
PSR 계산 완료: 185 레코드 (제거된 레코드: 2개)
수출 데이터와 결합 중...
최종 데이터 결합 완료: 197 레코드
   수출 데이터: 165개
   시가총액 데이터: 185개
   수출 예측치 포함: -20개
데이터 전처리 완료!
최종 데이터: 197 레코드
수출 예측치만 있는 미래 데이터: 12 레코드


In [17]:
from us_sarima_forecast import *

In [19]:
# 외생변수 포함 예측
forecast_result = sarima_forecast_with_export(
    final_data= result,
    export_forecast_start_date="2025-10",  # 매번 수동 입력
    USE_EXOGENOUS=True,
    forecast_months=12
)


SARIMA 예측 시작
예측 시작일: 2025-10
외생변수 사용: True
예측 기간: 12개월
예측 종료일: 2026-09
과거 수출 데이터: 153개
미래 수출 예측치: 12개
과거 PSR 데이터: 185개

외생변수(수출 데이터) 준비 중... (YoY 변환)


KeyError: "[Timestamp('2010-05-31 00:00:00'), Timestamp('2010-06-30 00:00:00'), Timestamp('2010-07-31 00:00:00'), Timestamp('2010-08-31 00:00:00'), Timestamp('2010-09-30 00:00:00'), Timestamp('2010-10-31 00:00:00'), Timestamp('2010-11-30 00:00:00'), Timestamp('2010-12-31 00:00:00'), Timestamp('2011-01-31 00:00:00'), Timestamp('2011-02-28 00:00:00'), Timestamp('2011-03-31 00:00:00'), Timestamp('2011-04-30 00:00:00'), Timestamp('2011-05-31 00:00:00'), Timestamp('2011-06-30 00:00:00'), Timestamp('2011-07-31 00:00:00'), Timestamp('2011-08-31 00:00:00'), Timestamp('2011-09-30 00:00:00'), Timestamp('2011-10-31 00:00:00'), Timestamp('2011-11-30 00:00:00'), Timestamp('2011-12-31 00:00:00'), Timestamp('2012-01-31 00:00:00'), Timestamp('2012-02-29 00:00:00'), Timestamp('2012-03-31 00:00:00'), Timestamp('2012-04-30 00:00:00'), Timestamp('2012-05-31 00:00:00'), Timestamp('2012-06-30 00:00:00'), Timestamp('2012-07-31 00:00:00'), Timestamp('2012-08-31 00:00:00'), Timestamp('2012-09-30 00:00:00'), Ti

In [7]:
# 현재 stock_prep 모듈에 어떤 함수들이 있는지 확인
print("stock_prep 모듈의 속성들:")
attributes = [attr for attr in dir(stock_prep) if not attr.startswith('_')]
for attr in attributes:
    print(f"- {attr}")

# 함수인지 확인
print("\n함수들:")
import types
functions = [attr for attr in dir(stock_prep) if isinstance(getattr(stock_prep, attr), types.FunctionType)]
for func in functions:
    print(f"- {func}()")


stock_prep 모듈의 속성들:
- STOCK_FUNCTION_AVAILABLE
- add_revenue_ttm
- calculate_psr_with_shift
- calendar
- collect_export_data
- collect_market_cap_data
- collect_revenue_data
- convert_to_month_end
- create_engine
- datetime
- fetch_market_data_yearly
- fetch_revenue_data
- get_db_host
- get_hs_data
- get_latest_input_date_data
- merge_revenue_market_data
- merge_with_export_data
- pd
- process_daily_to_monthly_market_data
- requests
- run_data_preprocessing
- run_simple_preprocessing
- test_api_connection
- time
- tqdm
- warnings

함수들:
- add_revenue_ttm()
- calculate_psr_with_shift()
- collect_export_data()
- collect_market_cap_data()
- collect_revenue_data()
- convert_to_month_end()
- create_engine()
- fetch_market_data_yearly()
- fetch_revenue_data()
- get_db_host()
- get_hs_data()
- get_latest_input_date_data()
- merge_revenue_market_data()
- merge_with_export_data()
- process_daily_to_monthly_market_data()
- run_data_preprocessing()
- run_simple_preprocessing()
- test_api_connectio